# BONITA — extrinsic evaluation on BoolQ (Italian)

*Francesco Baiocchi, Leonardo Petrilli — Task 3, course project work*

## Methodology

**Goal.** BONITA is an Italian version of Bonito (Nayak et al., 2024): a model that turns unannotated text into instruction-tuning data. Given a passage and a task type, it generates a task, i.e. an instruction with its input and the expected answer. Here the task type is boolean question answering: from a passage, BONITA writes a yes/no question about it and the answer (`Vero`/`Falso`).

**Why an extrinsic evaluation.** Comparing the generated questions with the reference ones by text similarity (an intrinsic evaluation) is of limited use: the task is one-to-many, since many different and correct questions can be asked about the same passage, so a low similarity does not necessarily mean a bad question. Following the Bonito paper, we therefore evaluate BONITA by what its data is used for: we instruction-tune a model on the tasks it generates and measure whether that model gets better on a benchmark with **human labels**. If the generated questions and answers are correct and useful, the model trained on them should answer real questions better than the model without that training.

**Pipeline.**

1. **Train BONITA** with QLoRA on `sapienzanlp/Minerva-7B-base-v1.0`, using 80% of `sapienzanlp/boolq_italian` (`train` + `validation`, split by passage, see Section 2) in the Bonito meta-template: the prompt contains the task type and the passage, the completion contains the question and the answer.
2. **Collect unannotated Italian text**: 10,000 paragraphs from Italian Wikipedia, the same source as the BoolQ passages. No label is attached to them.
3. **Generate a synthetic dataset**: BONITA writes one question/answer pair per paragraph. Generations that cannot be parsed and duplicate questions are discarded.
4. **Train the student**: a new LoRA on `sapienzanlp/Minerva-7B-instruct-v1.0`, trained **only** on the synthetic dataset.
5. **Evaluate** the baseline model (student without finetuning) and the student on the remaining 20% (2,490 human-labelled examples), never seen by BONITA.

**Controls.**

- *No leakage from the evaluation set.* The 80/20 split is done by passage, and the test examples whose passage or question also appears (as a near-duplicate) in BONITA's data are removed: neither BONITA nor the student ever sees a test passage or question. Model selection only uses dev sets held out from the respective training data.
- *Dev sets split by passage.* A passage can have several questions, so the held-out dev sets are formed by whole passages, and no passage is shared between train and dev.
- *A controlled comparison.* The student and the base model are the same 4-bit model and differ only by the LoRA adapter (the base model is evaluated with the adapter disabled). The prompts are identical, and the student is trained with the same prompt format used for evaluation. Any difference in the scores is therefore due to the fine-tuning on the synthetic data; note that this also includes adapting the model to the prompt format and to the `Vero`/`Falso` answers, not only what it learns from the content of the questions.
- *Reproducibility.* A single seed is used, generation is greedy, and every stage saves its output to disk, so that later stages can be re-run on exactly the same data.

**Evaluation protocol.** The answers are not generated as free text but scored by **log-likelihood**, as in `lm-evaluation-harness` (implemented directly in this notebook). For each question, the model computes the probability of the two continuations ` Vero` and ` Falso` after `Risposta:` and the more likely one is its answer. This removes any dependence on the output format and on answer parsing. The classes are unbalanced (61% `Vero`), so we report **accuracy**, **macro-F1** and the majority-class baseline that always answers `Vero`.

## 1. Setup & configuration

The whole pipeline runs on a single GPU. Models are handled with `transformers` (loading, generation), `peft` (LoRA adapters) and `trl` (`SFTTrainer`, supervised fine-tuning on prompt/completion pairs); datasets are loaded from the Hugging Face Hub with `datasets`.

In [1]:
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import gc
import json
import random
import re

import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset, concatenate_datasets, load_dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, EarlyStoppingCallback
from trl.trainer.sft_config import SFTConfig
from trl.trainer.sft_trainer import SFTTrainer

The configuration fixes the base model, the output file of each stage and the number of epochs. Every stage writes its output to disk (adapters, passages, generated tasks, results), so that the expensive stages do not have to be re-run to reproduce the later ones. A single seed is used everywhere.

In [ ]:
SEED = 42

BASE_MODEL = "sapienzanlp/Minerva-7B-base-v1.0"
INSTRUCTED_MODEL = "sapienzanlp/Minerva-7B-instruct-v1.0"

# outputs of each stage (each stage can be re-run from its files)
BONITA_DIR = "ckpt/bonita_full"  # generator adapter + tokenizer
WIKI_PATH = "wiki_passages.jsonl"  # unannotated passages
GENERATED_PATH = "generated_tasks.jsonl"  # parsed and filtered BONITA tasks
STUDENT_DIR = "ckpt/student_bonita"  # student adapter
RESULTS_DIR = "results"  # metrics and predictions (not versioned)
RESULTS_PATH = f"{RESULTS_DIR}/extrinsic_results.json"
PREDICTIONS_PATH = f"{RESULTS_DIR}/extrinsic_predictions.jsonl"

TEST_FRAC = 0.2  # share of BoolQ held out as the final test set of the base model and the student
N_PASSAGES = 10_000  # unannotated paragraphs given to BONITA (~ size of BoolQ train)

# for both, the checkpoint with the lowest loss on a held-out 5% of the training data is kept
# (evaluated every 100 steps)
BONITA_EPOCHS = 4
STUDENT_EPOCHS = 4


def free_memory() -> None:
    gc.collect()
    torch.cuda.empty_cache()


transformers.set_seed(SEED)  # seeds random, numpy and torch (CPU and CUDA)

The next cell defines the components shared by all the models of the pipeline.

**Quantization and LoRA.** All models are loaded in 4-bit NF4 with `bfloat16` compute, as in the previous notebooks. The base model is evaluated in the same quantized form as the student, so that the comparison only differs by the adapter. Every LoRA uses the configuration of BONITA: all linear projections, $r=16$, $\alpha=32$, dropout $0.05$.

Before adding a LoRA for training, the quantized model goes through `prepare_model_for_kbit_training` (PEFT), which in one call:

- freezes all the weights of the base model (only the LoRA is trained);
- casts the non-quantized parameters (layer norms, embeddings, `lm_head`) to `float32`, for numerical stability;
- enables gradient checkpointing, which recomputes the activations in the backward pass instead of storing them, to fit a 7B model with batch 8 in memory;
- enables the gradients on the input embeddings (`enable_input_require_grads`): with frozen embeddings and gradient checkpointing, the gradients would otherwise not flow back to the LoRA weights.

**Training arguments** (`sft_config`), the same for BONITA and the student: batch 8 with 2 accumulation steps (effective batch 16), learning rate $5\times10^{-5}$ with cosine decay and 3% warmup, `bfloat16`. The loss is computed on the completion only: the prompt (the passage) is given as input and is not a training target. The model is evaluated on a held-out dev set every 100 steps, and the checkpoint with the lowest dev loss is restored at the end of training. Training stops early if the dev loss does not improve for 3 evaluations in a row (early stopping).

**Dev split** (`split_by_group`): the dev set is a random 5% of the *passages*, not of the rows. Since a passage can have several questions, a row-level split would put the same passage in both train and dev, and the dev loss would be optimistic.

In [3]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)


def load_model(model_name: str = BASE_MODEL) -> transformers.PreTrainedModel:
    return AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        dtype=torch.bfloat16,
    )


def lora_config(trainable_token_ids: list[int] | None = None) -> LoraConfig:
    """LoRA on all linear projections, plus the rows of `trainable_token_ids` in the (untied) embeddings and LM head."""
    return LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules="all-linear",
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        trainable_token_indices={"embed_tokens": trainable_token_ids, "lm_head": trainable_token_ids}
        if trainable_token_ids
        else None,
    )


def sft_config(output_dir: str, num_epochs: int) -> SFTConfig:
    """Training arguments shared by BONITA and the student; the checkpoint with the lowest dev loss is kept."""
    return SFTConfig(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=2,
        learning_rate=5e-5,
        lr_scheduler_type="cosine",
        warmup_steps=0.03,  # float in [0, 1) = ratio of total steps
        bf16=True,
        completion_only_loss=True,
        eval_strategy="steps",  # dev loss every 100 steps
        eval_steps=100,
        save_strategy="steps",  # must match eval_strategy for load_best_model_at_end
        save_steps=100,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=25,
        seed=SEED,
    )


def split_by_group(rows: list[dict], key: str, frac: float, seed: int) -> tuple[list[dict], list[dict]]:
    """Splits rows into (train, held-out) so that all rows sharing `key` end up on the same side."""
    groups = sorted({r[key] for r in rows})
    random.Random(seed).shuffle(groups)
    held_out = set(groups[: round(len(groups) * frac)])
    return [r for r in rows if r[key] not in held_out], [r for r in rows if r[key] in held_out]


WORD_RE = re.compile(r"\w+")


def normalize(question: str) -> str:
    return " ".join(WORD_RE.findall(question.lower()))

## 2. Data: BoolQ splits and BONITA meta-template

`boolq_italian` has only `train` (9,399) and `validation` (3,259) splits, and 487 passages appear in both. The two splits are merged and divided 80/20: 80% (10,094 rows) is used to train BONITA, 20% is the final test set of the base model and the student.

The split is done by original (English) passage, so all the questions on a paragraph end up on the same side. BoolQ also contains near-duplicate paragraphs (e.g. two Wikipedia revisions of the same paragraph) with the same question, so the test examples whose Italian passage or question also appears in BONITA's data are removed (74 examples), leaving 2,490 test examples. A 5% slice of BONITA's passages is kept aside to monitor its dev loss.

Each BoolQ row becomes a prompt/completion pair:

```
<|tasktype|>
boolean question answering
<|context|>
{passage}
<|task|>                                              ← end of the prompt
{instruction} {question} Vero o Falso?
<|pipe|>
Risposta: Vero | Falso                                 ← end of the completion
```

The prompt contains the task type and the passage; the completion is the whole task, i.e. the instruction with the question, and after `<|pipe|>` the answer. BONITA therefore learns to generate both the question and its answer from a passage. The instruction is drawn at random from three Italian paraphrases, so that the generator does not always produce the same one.

In [5]:
boolq = load_dataset("sapienzanlp/boolq_italian")

boolq_all = [
    {**ex, "passage": ex["metadata"]["passage"]}
    for ex in concatenate_datasets([boolq["train"], boolq["validation"]]).to_list()
]
bonita_source, boolq_test = split_by_group(
    boolq_all, "passage", frac=TEST_FRAC, seed=SEED
)  # by original (English) passage

# near-duplicate paragraphs: drop the test examples whose Italian passage or question is also in BONITA's data
seen_passages = {ex["metadata"]["passage_translation"] for ex in bonita_source}
seen_questions = {normalize(ex["input_translation"]) for ex in bonita_source}
boolq_test = [
    ex
    for ex in boolq_test
    if ex["metadata"]["passage_translation"] not in seen_passages
    and normalize(ex["input_translation"]) not in seen_questions
]
print(f"BONITA source: {len(bonita_source)}, test: {len(boolq_test)}")

INSTRUCTION_VARIANTS = [
    "Dopo aver letto il passaggio, rispondi alla seguente domanda:",
    "In base al testo precedente, rispondi alla domanda:",
    "Secondo quanto riportato nel passaggio, rispondi alla domanda:",
]
QUESTION_SUFFIX = "Vero o Falso?"
SPECIAL_TOKENS = ["<|tasktype|>", "<|context|>", "<|task|>", "<|pipe|>"]

BONITA_INPUT = "<|tasktype|>\nboolean question answering\n<|context|>\n{passage}\n<|task|>"
BONITA_OUTPUT = "\n{instruction} {question}\n<|pipe|>\n{answer}"
PROMPT_TEMPLATE = "{passage}\n\n{task}\nRisposta:"  # student training and evaluation (Sections 6-7)


def answer_text(label: bool) -> str:
    return "Vero" if label else "Falso"


rng = random.Random(SEED)
bonita_rows = [
    {
        "prompt": BONITA_INPUT.format(passage=ex["metadata"]["passage_translation"]),
        "completion": BONITA_OUTPUT.format(
            instruction=rng.choice(INSTRUCTION_VARIANTS),
            question=f"{ex['input_translation']} {QUESTION_SUFFIX}",
            answer=f"Risposta: {answer_text(ex['label'])}",
        ),
        "passage": ex["passage"],
    }
    for ex in bonita_source
]
bonita_train, bonita_dev = split_by_group(bonita_rows, "passage", frac=0.05, seed=SEED)
print(f"BONITA train: {len(bonita_train)}, dev: {len(bonita_dev)}")
print(bonita_train[0]["prompt"] + bonita_train[0]["completion"])

BONITA source: 10094, test: 2490
BONITA train: 9611, dev: 483
<|tasktype|>
boolean question answering
<|context|>
Il persiano (/ˈpɜːrʒən, -ʃən/), conosciuto anche con il suo nome endonimo Farsi (فارسی fārsi (fɒːɾˈsiː) (ascolta)), è una delle lingue iraniche occidentali del ramo indo-iraniano della famiglia delle lingue indoeuropee. È parlato principalmente in Iran, Afghanistan (ufficialmente conosciuto come Dari dal 1958) e Tagikistan (ufficialmente conosciuto come Tajiki dall'era sovietica) e in alcune altre regioni che storicamente erano società persiane e considerate parte del Grande Iran. È scritto in alfabeto persiano, una variante modificata della scrittura araba, che a sua volta si è evoluta dall'alfabeto aramaico.
<|task|>
Secondo quanto riportato nel passaggio, rispondi alla domanda: L'Iran e l'Afghanistan parlano la stessa lingua? Vero o Falso?
<|pipe|>
Risposta: Vero


## 3. Training BONITA

QLoRA, lr $5\times10^{-5}$, cosine schedule, effective batch 16, loss on the completion only, for at most 4 epochs, with the dev loss evaluated every 100 steps.

The four markers of the meta-template (`<|tasktype|>`, `<|context|>`, `<|task|>`, `<|pipe|>`) are added to the tokenizer as special tokens, so that each one is a single, unsplittable token. The embedding matrix is resized only if it has no free rows left for them, which is the case for Minerva-7B-base (exactly 51,200 rows).

The rows added by the resize all start from the same mean vector, with a norm ~25 times smaller than that of a real token. Frozen, the four markers would be indistinguishable as input, and `<|pipe|>` could never win against a real token as output: BONITA would never close the question and would go on writing. For this reason:

- each new row (in the input embeddings and in the LM head, which are not tied in Minerva) is initialized as the mean of the rows of the token's name, e.g. `<|pipe|>` from ` pipe`, so the markers start distinct and with a typical norm;
- these 4 rows are trained together with the LoRA (`trainable_token_indices` of PEFT), while the rest of the embeddings stays frozen.

The adapter is saved with `save_embedding_layers=False`: only the LoRA and the 4 trained rows are stored, not the whole resized embeddings and LM head (~1.7 GB). The tokenizer is saved with it, since it is needed to reload BONITA with its special tokens.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.add_tokens(SPECIAL_TOKENS, special_tokens=True)

model = load_model()
if len(tokenizer) > model.config.vocab_size:
    model.resize_token_embeddings(len(tokenizer))

# each new row starts as the mean of the rows of the token's name (`<|pipe|>` -> ` pipe`), in both embeddings
special_token_ids = tokenizer.convert_tokens_to_ids(SPECIAL_TOKENS)
with torch.no_grad():
    for token, token_id in zip(SPECIAL_TOKENS, special_token_ids, strict=True):
        name_ids = tokenizer(" " + token.strip("<|>"), add_special_tokens=False).input_ids
        for layer in (model.get_input_embeddings(), model.get_output_embeddings()):
            layer.weight[token_id] = layer.weight[name_ids].mean(dim=0)

model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True
)  # freeze, fp32 norms, grad checkpointing
model = get_peft_model(model, lora_config(special_token_ids))  # LoRA + the 4 rows of the special tokens
model.print_trainable_parameters()

trainer = SFTTrainer(
    model=model,  # type: ignore
    args=sft_config("ckpt/outputs_bonita_full", BONITA_EPOCHS),
    train_dataset=Dataset.from_list(bonita_train).select_columns(["prompt", "completion"]),
    eval_dataset=Dataset.from_list(bonita_dev).select_columns(["prompt", "completion"]),
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],  # stop after 3 evaluations without improvement
)
trainer.train()
trainer.model.save_pretrained(BONITA_DIR, save_embedding_layers=False)  # LoRA + the 4 token rows only
tokenizer.save_pretrained(BONITA_DIR)

del trainer, model
free_memory()

## 4. Unannotated Italian text

BoolQ passages are Wikipedia paragraphs, so the unannotated text is taken from Italian Wikipedia (`wikimedia/wikipedia`, `20231101.it`). Only the first of its 10 parquet files is downloaded (~180k articles, far more than needed) and its articles are shuffled. From each article we take one paragraph of prose (ending with a full stop and at least as long as the 5th percentile of the length of BONITA's training passages in words, so that it is not much shorter than BoolQ passages), chosen at random.

The passages are collected only once and saved to `wiki_passages.jsonl`; if the file already exists it is loaded instead. To collect them again, delete the file.

In [ ]:
# one of the 10 files of Italian Wikipedia (~180k articles) is enough to sample the passages
WIKI_SHARD = "hf://datasets/wikimedia/wikipedia/20231101.it/train-00000-of-00010.parquet"
MIN_WORDS = int(np.percentile([len(ex["metadata"]["passage_translation"].split()) for ex in bonita_source], 5))


def candidate_paragraphs(text: str) -> list[str]:
    paragraphs = (p.strip() for p in text.split("\n"))
    return [p for p in paragraphs if p.endswith(".") and len(p.split()) >= MIN_WORDS]


def collect_wiki_passages() -> list[dict]:
    wiki = load_dataset("parquet", data_files=WIKI_SHARD, split="train").shuffle(seed=SEED)
    rng = random.Random(SEED)
    passages = []
    for article in wiki:
        candidates = candidate_paragraphs(article["text"])
        if candidates:
            passages.append({"passage": rng.choice(candidates)})
        if len(passages) >= N_PASSAGES:
            break
    return passages


# the passages are collected once
if os.path.exists(WIKI_PATH):
    print(f"loading the passages from {WIKI_PATH}")
    with open(WIKI_PATH) as f:
        wiki_passages = [json.loads(line) for line in f]
else:
    print(f"minimum paragraph length: {MIN_WORDS} words")
    wiki_passages = collect_wiki_passages()
    with open(WIKI_PATH, "w") as f:
        for row in wiki_passages:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"{len(wiki_passages)} passages")
print(wiki_passages[0])

## 5. Generating the instruction-tuning dataset with BONITA

BONITA generates one task per passage with greedy decoding (`do_sample=False`, for reproducibility); the other generation parameters are the defaults of Minerva-7B-instruct. Each generation is parsed into instruction + question and answer, and kept only if:

- it contains a parseable `Risposta: Vero/Falso` and the question ends with *"Vero o Falso?"*;
- the question is not a duplicate of an already kept question (after normalization).

The next cell reloads the base model with the BONITA adapter and generates in batches of 16 prompts, each made of the task type and a Wikipedia passage (the prompt part of the meta-template). The prompts of a batch have different lengths and are padded **on the left**: in this way every prompt ends right where generation starts, and the new tokens can be taken as everything after the input.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BONITA_DIR)
tokenizer.padding_side = "left"  # batched generation: every prompt must end right before the new tokens
base_model = load_model()
if len(tokenizer) > base_model.config.vocab_size:
    base_model.resize_token_embeddings(len(tokenizer))
model = PeftModel.from_pretrained(base_model, BONITA_DIR)
model.eval()


@torch.no_grad()
def batch_generate(model, tokenizer, prompts: list[str], batch_size: int = 16, **generation_kwargs) -> list[str]:
    """Generates one completion per prompt (new tokens only, special tokens removed)."""
    predictions = []
    for i in tqdm(range(0, len(prompts), batch_size)):
        inputs = tokenizer(prompts[i : i + batch_size], return_tensors="pt", padding=True).to(model.device)
        outputs = model.generate(
            **inputs, pad_token_id=tokenizer.pad_token_id, tokenizer=tokenizer, **generation_kwargs
        )
        new_tokens = outputs[:, inputs["input_ids"].shape[1] :]
        predictions += tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return predictions


generations = batch_generate(
    model,
    tokenizer,
    [BONITA_INPUT.format(passage=p["passage"]) for p in wiki_passages],
    max_new_tokens=128,
    stop_strings=["Risposta: Vero", "Risposta: Falso"],
    do_sample=False,
    use_cache=True,  # Minerva's config disables the KV cache
)

del model, base_model
free_memory()

**Parsing and filtering.** A generation is expected to look like `{instruction} {question} Vero o Falso? Risposta: Vero|Falso` (the `<|pipe|>` token is removed when decoding). The label is found with a regular expression on `Risposta: Vero/Falso`; everything before it is the *task* (instruction + question), which is what the student will see. The bare question (task without instruction and suffix) is only used for deduplication, after lowercasing and removing punctuation.

The kept tasks (passage, task, question, label) are saved to `generated_tasks.jsonl`, together with the counts of malformed and duplicate generations and the share of `Vero` labels, which shows whether BONITA's answers are balanced.

In [ ]:
LABEL_RE = re.compile(r"Risposta:\s*(Vero|Falso)\b", re.IGNORECASE)


def parse_generation(text: str) -> dict | None:
    """Splits a BONITA generation into task (instruction + question), question and label."""
    match = LABEL_RE.search(text)
    if match is None:
        return None
    task = text[: match.start()].strip()
    if not task.endswith(QUESTION_SUFFIX):
        return None
    question = task.removesuffix(QUESTION_SUFFIX)
    for variant in INSTRUCTION_VARIANTS:
        question = question.replace(variant, "")
    return {"task": task, "question": question.strip(), "label": match.group(1).capitalize() == "Vero"}


generated, seen = [], set()
counts = {"generated": len(generations), "malformed": 0, "duplicate": 0}
for passage, text in zip(wiki_passages, generations, strict=True):
    parsed = parse_generation(text)
    if parsed is None:
        counts["malformed"] += 1
        continue
    key = normalize(parsed["question"])
    if key in seen:
        counts["duplicate"] += 1
        continue
    seen.add(key)
    generated.append({"passage": passage["passage"], **parsed})

with open(GENERATED_PATH, "w") as f:
    for row in generated:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

counts["kept"] = len(generated)
counts["vero_rate"] = sum(r["label"] for r in generated) / len(generated)
print(counts)
for row in generated[:5]:
    print(row["passage"][:200], "...")
    print("  ->", row["task"], "|", answer_text(row["label"]))

## 6. Training the student

The student is a **fresh** LoRA on Minerva-7B-instruct, trained on the generated tasks only. Each task becomes a prompt/completion pair in the same format used for evaluation in Section 7:

```
{passage}

{instruction} {question} Vero o Falso?
Risposta:
```

with completion ` Vero` or ` Falso`. The loss is computed on the completion only. 5% of the training data is held out to pick the best checkpoint; no test example is used for training or model selection.

The completion starts with a space because the tokenizer encodes the word together with the preceding space (` Vero` is the single token `ĠVero`): the student is trained on exactly the tokens that are scored in Section 7. The next cell splits the rows by passage, loads a new 4-bit base model, adds a LoRA and trains it with the shared arguments of Section 1; at the end it prints the dev loss of each evaluation.

In [ ]:
student_rows = [
    {
        "prompt": PROMPT_TEMPLATE.format(passage=r["passage"], task=r["task"]),
        "completion": " " + answer_text(r["label"]),
        "passage": r["passage"],
    }
    for r in generated
]
student_train, student_dev = split_by_group(student_rows, "passage", frac=0.05, seed=SEED)
print(f"student train: {len(student_train)}, dev: {len(student_dev)}")
print(student_train[0]["prompt"] + student_train[0]["completion"])

transformers.set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(INSTRUCTED_MODEL)
model = prepare_model_for_kbit_training(load_model(INSTRUCTED_MODEL), use_gradient_checkpointing=True)  # see Section 1
model = get_peft_model(model, lora_config())

trainer = SFTTrainer(
    model=model,  # type: ignore
    args=sft_config("ckpt/outputs_student_bonita", STUDENT_EPOCHS),
    train_dataset=Dataset.from_list(student_train).select_columns(["prompt", "completion"]),
    eval_dataset=Dataset.from_list(student_dev).select_columns(["prompt", "completion"]),
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],  # stop after 3 evaluations without improvement
)
trainer.train()
trainer.save_model(STUDENT_DIR)
print(pd.DataFrame(trainer.state.log_history).dropna(subset=["eval_loss"])[["epoch", "eval_loss"]])

del trainer, model
free_memory()

## 7. Log-likelihood evaluation on the held-out 20%

Every test example is turned into the prompt of Section 6, with a fixed instruction (the first variant):

```
{passage}

Dopo aver letto il passaggio, rispondi alla seguente domanda: {question} Vero o Falso?
Risposta:
```

and the model scores the two continuations ` Vero` and ` Falso`: $\log P(\text{continuation} \mid \text{prompt})$ is the sum of the log-probabilities of the continuation tokens. The answer is the continuation with the highest score. We report the accuracy and the macro-F1, since the classes are unbalanced (61% `Vero`). The baseline that always answers `Vero` (majority class) is shown for reference.

**Implementation** (next cell):

- `encode_request` tokenizes prompt and continuation separately and concatenates them, so the prompt tokens are the same for both continuations and the continuation starts right after them;
- `continuation_logprobs` pads the requests of a batch on the right and runs one forward pass per batch. The logits at position $t-1$ give the distribution of the token at position $t$, so the log-probabilities of the continuation tokens are read from the positions just before them and summed;
- `evaluate_model` scores both continuations for every example; the index of the best one is directly the predicted label, since `CHOICES` is ordered as `[Falso, Vero]`.

In [6]:
CHOICES = [" Falso", " Vero"]  # index = gold label (False = 0, True = 1)

eval_examples = [
    {
        "id": ex["id"],
        "prompt": PROMPT_TEMPLATE.format(
            passage=ex["metadata"]["passage_translation"],
            task=f"{INSTRUCTION_VARIANTS[0]} {ex['input_translation']} {QUESTION_SUFFIX}",
        ),
        "label": int(ex["label"]),
    }
    for ex in boolq_test
]
gold_labels = np.array([ex["label"] for ex in eval_examples])
print(len(eval_examples), "test examples")
print(eval_examples[0]["prompt"])


def encode_request(tokenizer, context: str, continuation: str) -> tuple[list[int], int]:
    """Returns the token ids of context + continuation and the number of context tokens."""
    context_ids = tokenizer(context).input_ids
    return context_ids + tokenizer(continuation, add_special_tokens=False).input_ids, len(context_ids)


@torch.no_grad()
def continuation_logprobs(
    model, tokenizer, contexts: list[str], continuations: list[str], batch_size: int = 16
) -> np.ndarray:
    """Sum of the log-probabilities of each continuation given its context."""
    requests = [encode_request(tokenizer, c, x) for c, x in zip(contexts, continuations, strict=True)]
    scores = np.zeros(len(requests))
    for start in tqdm(range(0, len(requests), batch_size)):
        batch = range(start, min(start + batch_size, len(requests)))
        max_len = max(len(requests[i][0]) for i in batch)
        input_ids = torch.full((len(batch), max_len), tokenizer.pad_token_id, dtype=torch.long)
        attention_mask = torch.zeros_like(input_ids)
        for row, i in enumerate(batch):  # right padding: positions are unchanged
            ids = requests[i][0]
            input_ids[row, : len(ids)] = torch.tensor(ids)
            attention_mask[row, : len(ids)] = 1
        logits = model(input_ids=input_ids.to(model.device), attention_mask=attention_mask.to(model.device)).logits
        for row, i in enumerate(batch):
            ids, n_context = requests[i]
            # the token at position t is predicted by the logits at position t - 1
            logprobs = torch.log_softmax(logits[row, n_context - 1 : len(ids) - 1].float(), dim=-1)
            targets = torch.tensor(ids[n_context:], device=logprobs.device)
            scores[i] = logprobs.gather(-1, targets[:, None]).sum().item()
    return scores


def evaluate_model(model, tokenizer) -> dict[str, np.ndarray]:
    contexts = [ex["prompt"] for ex in eval_examples for _ in CHOICES]
    continuations = [c for _ in eval_examples for c in CHOICES]
    loglik = continuation_logprobs(model, tokenizer, contexts, continuations).reshape(-1, len(CHOICES))
    return {"loglik": loglik, "pred": loglik.argmax(axis=1)}

2490 test examples
Now You See Me è una serie di film di genere thriller e colpi scritti da Ed Solomon, Boaz Yakin e Edward Ricourt. I film si concentrano sulle azioni di un team di illusionisti chiamati ``The Four Horsemen'' che portano a termine colpi quasi impossibili. La serie presenta un cast di attori tra cui Jesse Eisenberg, Mark Ruffalo, Woody Harrelson, Isla Fisher, Dave Franco, Michael Caine, Lizzy Caplan e Morgan Freeman. Il primo film è stato rilasciato nel 2013, mentre il secondo è stato rilasciato nel 2016 e un terzo film è attualmente in fase di sviluppo e dovrebbe essere rilasciato nel 2019. La serie ha ricevuto recensioni miste da critici e pubblico e ha incassato quasi 700 milioni di dollari in tutto il mondo.

Dopo aver letto il passaggio, rispondi alla seguente domanda: Ci sarà un terzo capitolo di Now You See Me? Vero o Falso?
Risposta:


The base model and the student are evaluated on the same 4-bit base model: the base score is computed with the adapter disabled.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(INSTRUCTED_MODEL)
base_model = load_model(INSTRUCTED_MODEL)
model = PeftModel.from_pretrained(base_model, STUDENT_DIR)
model.eval()

outputs = {}
with model.disable_adapter():
    outputs["Minerva-7B-instruct (baseline)"] = evaluate_model(model, tokenizer)
outputs["student (BONITA data)"] = evaluate_model(model, tokenizer)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  0%|          | 0/312 [00:00<?, ?it/s]

  0%|          | 0/312 [00:00<?, ?it/s]

## 8. Results

For each model, accuracy and macro-F1 (the mean of the F1 of the two classes) are computed from the predictions of Section 7. The majority baseline answers `Vero` to every question: its accuracy is the share of `Vero` in the test set, while its macro-F1 is low, because the `Falso` class is never predicted.

In [8]:
def macro_f1(pred: np.ndarray, gold: np.ndarray) -> float:
    f1s = []
    for c in (0, 1):
        tp = np.sum((pred == c) & (gold == c))
        fp = np.sum((pred == c) & (gold != c))
        fn = np.sum((pred != c) & (gold == c))
        f1s.append(2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0)
    return float(np.mean(f1s))


def summarize(pred: np.ndarray) -> dict[str, float]:
    return {"acc": float(np.mean(pred == gold_labels)), "macro_f1": macro_f1(pred, gold_labels)}


results = {"always Vero (majority)": summarize(np.ones_like(gold_labels))}
for name, out in outputs.items():
    results[name] = summarize(out["pred"])

results_table = pd.DataFrame(results).T
display(results_table)

,acc,macro_f1
always Vero (majority),0.609639,0.378743
Minerva-7B-instruct (base),0.797189,0.794341
student (BONITA data),0.884739,0.877809


The metrics and the generation statistics are saved to `results/extrinsic_results.json`; the log-likelihoods and predictions of every test example are saved to `results/extrinsic_predictions.jsonl`, so that the analysis can be repeated without running the models again.

In [9]:
os.makedirs(RESULTS_DIR, exist_ok=True)
with open(RESULTS_PATH, "w") as f:
    json.dump(
        {
            "generation": counts if "counts" in globals() else None,
            "results": results,
        },
        f,
        indent=1,
    )

with open(PREDICTIONS_PATH, "w") as f:
    for i, ex in enumerate(eval_examples):
        row = {"id": ex["id"], "label": ex["label"]}
        for name, out in outputs.items():
            row[name] = {"loglik": out["loglik"][i].tolist(), "pred": int(out["pred"][i])}
        f.write(json.dumps(row, ensure_ascii=False) + "\n")